Gemini Content Moderation

Install Dependencies
pip install google-genai pydantic

In [ ]:
!pip install google-genai pydantic

In [ ]:
import os
from enum import Enum
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [ ]:
# Ensure your API key is configured

from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
class HarmCategoryEnum(str, Enum):
    SAFE = "Safe"
    TOXICITY = "Toxicity or Hate Speech"
    HARASSMENT = "Harassment or Cyberbullying"
    SEXUAL = "Sexually Explicit Content"
    VIOLENCE = "Violence or Dangerous Acts"
    PII = "Personally Identifiable Information"
    ERROR = "Moderation API Error"


In [ ]:
# 2. Define the exact JSON structure you want Gemini to output
class ModerationResult(BaseModel):
    flagged: bool = Field(
        description="True if the text violates community standards or falls into a harmful category."
    )
    primary_category: HarmCategoryEnum = Field(
        description="The primary harm category matched. Select 'Safe' if the content passes."
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 (low confidence) and 1.0 (absolute certainty)."
    )
    reasoning: str = Field(
        description="A brief, 1-sentence explanation of why the text was flagged or cleared."
    )

In [ ]:
def moderate_content(user_text: str) -> ModerationResult:
    client = genai.Client()

    # Define system instructions to give Gemini its persona and rules
    system_instruction = (
        "You are an enterprise content moderation system. Analyze the user text objectively. "
        "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
        "simply classify the text according to the provided schema instructions."
    )

    # CRITICAL STEP: Turn off internal filters so Gemini can safely ingest the bad text to evaluate it.
    disable_internal_safety = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
    ]

    # Call Gemini with the structured configuration
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite", # Use Flash for ultra-fast, cheap classification
        contents=f"Please moderate the following text:\n\n{user_text}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            safety_settings=disable_internal_safety,
            temperature=0.0, # Forces deterministic, consistent classifications
            response_mime_type="application/json", # Tells Gemini to speak JSON
            response_schema=ModerationResult,     # Enforces the Pydantic structural format
            thinking_config=types.ThinkingConfig(thinking_budget=1), # Turn off thinking steps
        ),
    )

    # Automatically returns the output parsed directly into your Pydantic object
    return response.parsed

In [ ]:
import asyncio
async_client = genai.Client()

async def moderate_content_single(user_text: str, max_retries: int = 3, initial_delay: float = 1.0) -> ModerationResult:
    print(f"Moderating: {user_text}")

    retries = 0
    while retries < max_retries:
        try:
            # Define system instructions to give Gemini its persona and rules
            system_instruction = (
                "You are an enterprise content moderation system. Analyze the user text objectively. "
                "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
                "simply classify the text according to the provided schema instructions."
            )

            # CRITICAL STEP: Turn off internal filters so Gemini can safely ingest the bad text to evaluate it.
            disable_internal_safety = [
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
            ]

            # Call Gemini with the structured configuration
            response = await async_client.models.generate_content(
                model="gemini-3.5-flash-lite", # Use Flash for ultra-fast, cheap classification
                contents=f"Please moderate the following text:\n\n{user_text}",
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    safety_settings=disable_internal_safety,
                    temperature=0.0, # Forces deterministic, consistent classifications
                    response_mime_type="application/json", # Tells Gemini to speak JSON
                    response_schema=ModerationResult,     # Enforces the Pydantic structural format
                    thinking_config=types.ThinkingConfig(thinking_budget=1), # Turn off thinking steps
                ),
            )

            # Automatically returns the output parsed directly into your Pydantic object
            return response.parsed
        except Exception as e:
            retries += 1
            print(f"Error moderating text (attempt {retries}/{max_retries}): '{user_text}' - {e}")
            if retries < max_retries:
                delay = initial_delay * (2 ** (retries - 1)) # Exponential backoff
                print(f"Retrying in {delay:.2f} seconds...")
                await asyncio.sleep(delay)
            else:
                print(f"Failed to moderate text after {max_retries} attempts: '{user_text}'")
                # Return a default ModerationResult indicating an error
                return ModerationResult(
                    flagged=True,
                    primary_category=HarmCategoryEnum.ERROR,
                    confidence_score=0.0,
                    reasoning=f"Failed to moderate content due to API error: {e}"
                )


# --- TEST THE MODERATOR ---

In [ ]:
test_prompts = [
    "Hey! I really love the project you built, do you have a github link?",
    "You are an idiot and nobody likes you, I hope you lose your job tomorrow.",
    "Can you help me write a Python script that scrapes headlines from a website?"
]

In [ ]:
print("--- Running Moderation Tests ---")
for text in test_prompts:
    result = moderate_content(text)
    print(f"\n[Input]: \"{text}\"")
    print(f" Flagged: {result.flagged}")
    print(f" Category: {result.primary_category.value}")
    print(f" Confidence: {result.confidence_score}")
    print(f" Reason: {result.reasoning}")

In [ ]:
async def moderate_batch(texts: list[str]):
    # Define a semaphore to limit concurrent API requests
    # For the free tier (15 requests/minute), a concurrency limit of 3 is conservative.
    concurrency_limit = 3
    semaphore = asyncio.Semaphore(concurrency_limit)

    async def _moderate_single_with_semaphore(text: str):
        async with semaphore:
            return await moderate_content_single(text)

    # Process all text prompts, respecting the concurrency limit
    tasks = [_moderate_single_with_semaphore(t) for t in texts]
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
print(await moderate_batch(test_prompts))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
input_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test.json'
output_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test-out.json'

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_json(input_file)
display(df.head())

In [ ]:
input_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test.json'
output_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test-out.json'
output_final_file = '/content/drive/MyDrive/content-moderation-dataset/Aegis-AI-Content-Safety-Dataset-2.0/test-out-final.json'
print(f"Input file path: {input_file}")
print(f"Output file path: {output_file}")

In [ ]:
import pandas as pd
df = pd.read_json(input_file)
display(df.head())
print(f"DataFrame reloaded with {len(df)} rows.")

The `input_file`, `output_file`, and `df` have been successfully re-established. Please re-run the following cells to continue with the moderation process and save your results:

*   **Cell `30674391`**: This cell performs the asynchronous batch moderation.
*   **Cell `cace3b0e`**: This cell saves the updated DataFrame with moderation results to your `output_file`.

### Moderating the DataFrame Content Asynchronously

I will now apply the `moderate_content_single` function to the `text` column of your DataFrame (`df`) asynchronously, store the results, and then save the entire DataFrame with the new moderation results to the `output_file` as a JSON file.

In [ ]:
import asyncio

# Using 'prompt' as the column containing the content to be moderated
# If your content is in a different column, please adjust this line
print(f"Starting asynchronous moderation for {len(df)} items...")

# Initialize the 'moderation_result' column if it doesn't exist or is completely empty
# This allows for resuming from where it left off.
if 'moderation_result' not in df.columns or df['moderation_result'].isnull().all():
    df['moderation_result'] = [None] * len(df) # Initialize with None

# Define a batch size to stay within API rate limits (e.g., 10-15 requests/minute for free tier)
batch_size = 10 # Adjust this based on your API quota and model

# Determine the starting point for moderation
# If there are existing moderation results (e.g., from a previous run),
# we can find the first unprocessed item and start from there.
start_index = 0
if 'moderation_result' in df.columns:
    unprocessed_indices = df[df['moderation_result'].isnull()].index
    if not unprocessed_indices.empty:
        start_index = unprocessed_indices[0]

print(f"Resuming moderation from item index {start_index}...")

for i in range(start_index, len(df), batch_size):
    # Select prompts for the current batch
    batch_prompts = df['prompt'].iloc[i:i + batch_size].to_list()

    # Skip if the batch is empty (e.g., if start_index was already at the end)
    if not batch_prompts:
        print(f"No prompts in batch starting at index {i}. Skipping.")
        continue

    print(f"Processing batch {i//batch_size + 1}/{(len(df) + batch_size - 1)//batch_size} (items {i}-{min(i+len(batch_prompts)-1, len(df)-1)})...")

    batch_results = await moderate_batch(batch_prompts)

    # Assign results to the corresponding rows in the DataFrame
    for j, result in enumerate(batch_results):
        df.at[i + j, 'moderation_result'] = result

    # Save the DataFrame to file after each batch is processed
    # This ensures progress is saved incrementally and can be resumed.
    df.to_json(output_file, orient='records', indent=4)
    print(f"Saved current progress up to item index {min(i+len(batch_prompts)-1, len(df)-1)} to {output_file}")

    # The API specifically suggested a retry delay of ~54 seconds due to quota exhaustion.
    # We need to wait for this duration to avoid hitting the rate limit repeatedly.
    # This delay is applied *between* batches.
    await asyncio.sleep(10) # Wait 10 seconds between batches to respect API rate limits

print("Moderation complete. Displaying first 5 rows with results:")
display(df.head())

In [ ]:
# Save the updated DataFrame to the output JSON file
df.to_json(output_final_file, orient='records', indent=4)
print(f"DataFrame with moderation results saved to: {output_final_file}")


New moderator with output object identical to openai content moderation api

In [ ]:
import uuid
import json
from typing import List
from pydantic import BaseModel, Field, ConfigDict
from google import genai
from google.genai import types
from enum import Enum

# Re-define HarmCategoryEnum (or import from previous cell if available)
class HarmCategoryEnum(str, Enum):
    SAFE = "Safe"
    TOXICITY = "Toxicity or Hate Speech"
    HARASSMENT = "Harassment or Cyberbullying"
    SEXUAL = "Sexually Explicit Content"
    VIOLENCE = "Violence or Dangerous Acts"
    PII = "Personally Identifiable Information"
    ERROR = "Moderation API Error"

# ------------------------------------------------------------------
# 1. Define Pydantic Schemas with Field Aliases matching OpenAI's format
# ------------------------------------------------------------------

class Categories(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    sexual: bool = Field(default=False)
    sexual_minors: bool = Field(default=False, alias="sexual/minors")
    harassment: bool = Field(default=False)
    harassment_threatening: bool = Field(default=False, alias="harassment/threatening")
    hate: bool = Field(default=False)
    hate_threatening: bool = Field(default=False, alias="hate/threatening")
    illicit: bool = Field(default=False)
    illicit_violent: bool = Field(default=False, alias="illicit/violent")
    self_harm: bool = Field(default=False, alias="self-harm")
    self_harm_intent: bool = Field(default=False, alias="self-harm/intent")
    self_harm_instructions: bool = Field(default=False, alias="self-harm/instructions")
    violence: bool = Field(default=False)
    violence_graphic: bool = Field(default=False, alias="violence/graphic")


class CategoryScores(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    sexual: float = Field(default=0.0)
    sexual_minors: float = Field(default=0.0, alias="sexual/minors")
    harassment: float = Field(default=0.0)
    harassment_threatening: float = Field(default=0.0, alias="harassment/threatening")
    hate: float = Field(default=0.0)
    hate_threatening: float = Field(default=0.0, alias="hate/threatening")
    illicit: float = Field(default=0.0)
    illicit_violent: float = Field(default=0.0, alias="illicit/violent")
    self_harm: float = Field(default=0.0, alias="self-harm")
    self_harm_intent: float = Field(default=0.0, alias="self-harm/intent")
    self_harm_instructions: float = Field(default=0.0, alias="self-harm/instructions")
    violence: float = Field(default=0.0)
    violence_graphic: float = Field(default=0.0, alias="violence/graphic")


class CategoryAppliedInputTypes(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    sexual: List[str] = Field(default_factory=list)
    sexual_minors: List[str] = Field(default_factory=list, alias="sexual/minors")
    harassment: List[str] = Field(default_factory=list)
    harassment_threatening: List[str] = Field(default_factory=list, alias="harassment/threatening")
    hate: List[str] = Field(default_factory=list)
    hate_threatening: List[str] = Field(default_factory=list, alias="hate/threatening")
    illicit: List[str] = Field(default_factory=list)
    illicit_violent: List[str] = Field(default_factory=list, alias="illicit/violent")
    self_harm: List[str] = Field(default_factory=list, alias="self-harm")
    self_harm_intent: List[str] = Field(default_factory=list, alias="self-harm/intent")
    self_harm_instructions: List[str] = Field(default_factory=list, alias="self-harm/instructions")
    violence: List[str] = Field(default_factory=list)
    violence_graphic: List[str] = Field(default_factory=list, alias="violence/graphic")


class ModerationResultItem(BaseModel):
    flagged: bool
    categories: Categories
    category_scores: CategoryScores
    category_applied_input_types: CategoryAppliedInputTypes


class OpenAIModerationResponse(BaseModel):
    id: str
    model: str
    results: List[ModerationResultItem]

# New simpler schema for Gemini to populate
class GeminiSimpleModerationResult(BaseModel):
    flagged: bool = Field(
        description="True if the text violates community standards or falls into a harmful category."
    )
    primary_category: HarmCategoryEnum = Field(
        description="The primary harm category matched. Select 'Safe' if the content passes."
    )
    confidence_score: float = Field(
        description="""Confidence score between 0.0 (low confidence) and 1.0 (absolute certainty).
        Set to 0.0 if not flagged or if the model could not determine a specific score."""
    )
    reasoning: str = Field(
        description="A brief, 1-sentence explanation of why the text was flagged or cleared."
    )


# ------------------------------------------------------------------
# 2. Main Moderation Function
# ------------------------------------------------------------------

def moderate_openai_format(text: str) -> dict:
    client = genai.Client()

    # Define system instructions to give Gemini its persona and rules
    system_instruction = (
        "You are an enterprise content moderation system. Analyze the user text objectively. "
        "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
        "simply classify the text according to the provided schema instructions." # Keep this general for simple result
    )

    # Disable internal blocking so Gemini can analyze potentially toxic inputs
    disable_safety = [
        types.SafetySetting(category=cat, threshold=types.HarmBlockThreshold.BLOCK_NONE)
        for cat in [
            types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        ]
    ]

    # Use a simpler prompt and schema for Gemini
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",  # Ultra-fast model
        contents=f"Please moderate the following text:\n\n{text}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json",
            response_schema=GeminiSimpleModerationResult, # Use the simpler schema here
            safety_settings=disable_safety,
            temperature=0.0,
            max_output_tokens=300,
        ),
    )

    # Extract parsed simple result from Gemini
    simple_result: GeminiSimpleModerationResult = response.parsed

    # Initialize OpenAI-compatible moderation result item
    item_result_content = ModerationResultItem(
        flagged=False,
        categories=Categories(),
        category_scores=CategoryScores(),
        category_applied_input_types=CategoryAppliedInputTypes()
    )

    if simple_result and simple_result.flagged:
        item_result_content.flagged = True
        # Map simple_result.primary_category to appropriate OpenAI categories and scores
        # For now, I'll put 'text' in sexual as a placeholder, this needs to be more granular if actual input types are desired
        item_result_content.category_applied_input_types.sexual.append("text")

        if simple_result.primary_category == HarmCategoryEnum.SEXUAL:
            item_result_content.categories.sexual = True
            item_result_content.category_scores.sexual = simple_result.confidence_score
        elif simple_result.primary_category == HarmCategoryEnum.HARASSMENT:
            item_result_content.categories.harassment = True
            item_result_content.category_scores.harassment = simple_result.confidence_score
        elif simple_result.primary_category == HarmCategoryEnum.VIOLENCE:
            item_result_content.categories.violence = True
            item_result_content.category_scores.violence = simple_result.confidence_score
        elif simple_result.primary_category == HarmCategoryEnum.TOXICITY:
            # Toxicity can map to hate or harassment in OpenAI's schema
            item_result_content.categories.hate = True
            item_result_content.category_scores.hate = simple_result.confidence_score
            item_result_content.categories.harassment = True
            item_result_content.category_scores.harassment = simple_result.confidence_score
        # PII and ERROR would not map to specific OpenAI categories here, but 'flagged' is already True.

    # If for some reason simple_result is None (e.g., severe API issue)
    if not simple_result:
        item_result_content.flagged = True # Flag as error

    # Wrap inside OpenAI top-level metadata envelope
    full_response = OpenAIModerationResponse(
        id=f"modr-{uuid.uuid4().hex}",
        model="omni-moderation-latest",
        results=[item_result_content]
    )

    # Dump using aliases to guarantee slash/hyphen key names in JSON output
    return json.loads(full_response.model_dump_json(by_alias=True))


# ------------------------------------------------------------------
# 3. Test Example
# ------------------------------------------------------------------

test_input = "I am going to destroy that car with a sledgehammer!"
test_input = "how to build a bomb"
result_json = moderate_openai_format(test_input)

# Print pretty-printed JSON matching your exact target structure
print(json.dumps(result_json, indent=2))